# Chuẩn bị dữ liệu đã gán nhãn

Đồng bộ thân bài từ dữ liệu clean và xác định những bài đủ điều kiện huấn luyện. Nhãn thủ công, trạng thái, ghi chú, ticker và thứ tự dòng được giữ nguyên.

Bộ nhãn huấn luyện hiện có 7 nhóm: `macro` được quy về `industry` khi ánh xạ, không ghi đè nhãn gốc.

## 1. Thiết lập

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

In [2]:
CURRENT_DIR = Path.cwd().resolve()
PROJECT_CANDIDATES = [CURRENT_DIR / 'news_processing', CURRENT_DIR, *CURRENT_DIR.parents]
PROJECT_DIR = next(
    (path for path in PROJECT_CANDIDATES
     if (path / 'src').is_dir() and (path / 'data').is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Không tìm thấy thư mục news_processing.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

In [3]:
from src.annotation_utils import (
    normalize_news_type,
    refresh_annotation_content,
    save_annotation_csv,
)

ANNOTATION_PATH = PROJECT_DIR / 'data/annotation/news_type/news_manually_labeled.csv'
CLEAN_PATH = PROJECT_DIR / 'data/processed/news_clean.csv'
EXCLUDED_PATH = PROJECT_DIR / 'data/interim/news_excluded_records.csv'
MERGE_LOG_PATH = PROJECT_DIR / 'data/interim/duplicate_merge_log.csv'
LABEL_MAPPING_PATH = PROJECT_DIR / 'models/news_type_phobert_cnn/label_mapping.json'

## 2. Đọc dữ liệu

Đọc bản nhãn cùng log loại trừ và hợp nhất để đối chiếu theo ID, không ghép theo vị trí dòng.

In [4]:
annotations = pd.read_csv(ANNOTATION_PATH, dtype=str, keep_default_na=False)
clean_news = pd.read_csv(CLEAN_PATH, dtype=str, keep_default_na=False)
excluded_records = pd.read_csv(EXCLUDED_PATH, dtype=str, keep_default_na=False)
duplicate_merge_log = pd.read_csv(MERGE_LOG_PATH, dtype=str, keep_default_na=False)
label_mapping = json.loads(LABEL_MAPPING_PATH.read_text(encoding='utf-8-sig'))

print('Bài trong file annotation:', len(annotations))
print('Bài trong dữ liệu clean:', len(clean_news))
display(annotations['annotation_status'].value_counts().rename('article_count').to_frame())

Bài trong file annotation: 1200
Bài trong dữ liệu clean: 3356


,article_count
annotation_status,
labeled,1111
review_required,87
excluded,2


## 3. Đồng bộ thân bài

Những bài thiếu, sai hoặc chưa xác minh được nội dung vẫn giữ nhãn để truy vết, nhưng để trống `content` và chưa dùng để huấn luyện.

In [5]:
updated_annotations, annotation_audit = refresh_annotation_content(
    annotations, clean_news, excluded_records, duplicate_merge_log,
)

blocked_articles = annotation_audit.loc[annotation_audit['content_issue'].ne('')]
print('Bài được cập nhật content:', int(annotation_audit['content_changed'].sum()))
print('Bài cần xử lý lại nội dung:', len(blocked_articles))
display(blocked_articles['content_issue'].value_counts().rename('article_count').to_frame())

Bài được cập nhật content: 0
Bài cần xử lý lại nội dung: 104


,article_count
content_issue,
CONTENT_QUALITY:UNRESOLVED_TRAILING_FRAGMENT,54
CONTENT_QUALITY:RELATED_LINKS_WITHOUT_BODY,33
CONTENT_QUALITY:CONTENT_TOO_SHORT|SUMMARY_ONLY,7
CONTENT_QUALITY:CONTENT_TOO_SHORT|INFOGRAPHIC_WITHOUT_TEXT|TITLE_ONLY,3
CONTENT_QUALITY:MISMATCHED_BODY,3
CONTENT_QUALITY:CATEGORY_PAGE,2
CONTENT_QUALITY:PODCAST_WITHOUT_TRANSCRIPT,1
CONTENT_QUALITY:SUMMARY_ONLY,1


Trong 104 bài cần xử lý nội dung, 54 bài chỉ có đoạn cuối chưa rõ ranh giới. Đây là cờ nghi vấn, không phải kết luận toàn bộ bài sai.

## 4. Kiểm tra bảo toàn nhãn

Kiểm tra các trường được bảo vệ trước khi lưu. Việc quy đổi nhãn chỉ phục vụ dữ liệu huấn luyện.

In [6]:
protected_columns = annotations.columns.difference(['content'], sort=False)
normalized_labels = updated_annotations['news_type'].map(normalize_news_type)
valid_labels = normalized_labels.eq('') | normalized_labels.isin(label_mapping['label2id'])

checks = {
    'Giữ đủ bài và thứ tự news_id': updated_annotations['news_id'].equals(annotations['news_id']),
    'Không trùng news_id': updated_annotations['news_id'].is_unique,
    'Chỉ thay đổi content': updated_annotations[protected_columns].equals(annotations[protected_columns]),
    'Nhãn thuộc bộ nhãn đã chốt': valid_labels.all(),
    'Trạng thái hợp lệ': updated_annotations['annotation_status'].isin(
        ['labeled', 'review_required', 'excluded']
    ).all(),
    'Bài lỗi nội dung không vào tập đủ điều kiện': not annotation_audit.loc[
        annotation_audit['content_issue'].ne(''), 'training_eligible'
    ].any(),
}
display(pd.Series(checks, name='passed').to_frame())
assert all(checks.values()), 'Dừng lưu: kiểm tra bảo toàn nhãn chưa đạt.'

,passed
Giữ đủ bài và thứ tự news_id,True
Không trùng news_id,True
Chỉ thay đổi content,True
Nhãn thuộc bộ nhãn đã chốt,True
Trạng thái hợp lệ,True
Bài lỗi nội dung không vào tập đủ điều kiện,True


## 5. Những bài đủ điều kiện huấn luyện

In [7]:
eligible_mask = annotation_audit['training_eligible']
training_label_counts = (
    annotation_audit.loc[eligible_mask, 'training_news_type']
    .value_counts().rename('article_count')
)

print('Bài đủ điều kiện:', int(eligible_mask.sum()))
print('Bài chưa dùng để huấn luyện:', int((~eligible_mask).sum()))
display(training_label_counts.to_frame())

Bài đủ điều kiện: 1058
Bài chưa dùng để huấn luyện: 142


,article_count
training_news_type,
business_operations,298
stock_market,204
financial_results,178
governance_capital,151
mention_only,119
industry,58
legal_compliance,50


Có 1.058 bài đủ điều kiện và 142 bài chưa dùng để huấn luyện. Trong nhóm chưa đủ điều kiện, 104 bài cần xử lý nội dung; 38 bài còn nội dung sử dụng được nhưng chưa chốt nhãn.

Phân bố vẫn lệch: `business_operations` có 298 bài, còn `legal_compliance` có 50 bài. Gộp nhãn không đồng nghĩa dữ liệu đã cân bằng.

## 6. Lưu bản nhãn

Chỉ lưu khi bảng kiểm tra đạt yêu cầu. Hàm xuất sẽ từ chối nếu thao tác làm mất hoặc thay đổi nhãn đã nhập.

In [ ]:
save_annotation_csv(updated_annotations, ANNOTATION_PATH)
print('Đã lưu:', ANNOTATION_PATH.relative_to(PROJECT_DIR).as_posix())